In [2]:
import os
from getpass import getpass
from pathlib import Path
from langchain_community.document_loaders import (
    PyMuPDFLoader,
    TextLoader,
    Docx2txtLoader,
    DirectoryLoader
)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import FastEmbedSparse
from langchain_qdrant import RetrievalMode
import pandas as pd

In [6]:
DATA_DIR = Path("../data")

In [ ]:
if not os.getenv("OPENAI_API_KEY_EMBEDDINGS"):
    os.environ["OPENAI_API_KEY_EMBEDDINGS"] = getpass("Enter your RCP API key for embedding model: ")

embeddings = OpenAIEmbeddings(
    model="Qwen/Qwen3-Embedding-8B",
    base_url="https://inference.rcp.epfl.ch/v1",
    api_key=os.getenv("OPENAI_API_KEY_EMBEDDINGS")
)

In [7]:
# TODO : tester UnstructuredWordDocumentLoader (permet de conserver des éléments séparés avec mode="elements") et DoclingLoader

loaders = [
    DirectoryLoader(DATA_DIR, glob="**/*.pdf", loader_cls=PyMuPDFLoader, loader_kwargs={"mode": "single"}, show_progress=True),
    DirectoryLoader(DATA_DIR, glob="**/*.txt", loader_cls=TextLoader, show_progress=True),
    DirectoryLoader(DATA_DIR, glob="**/*.docx", loader_cls=Docx2txtLoader, show_progress=True),
]

In [8]:
docs = []
for loader in loaders:
    docs.extend(loader.load())

# TODO : tester avec d'autres parametres de chunking ou voir https://docs.langchain.com/oss/python/integrations/splitters
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    #separators=["\n \n", "\n\n", "\n", ". ", " ", ""], TODO : a tester
    add_start_index=True,
)
chunks = splitter.split_documents(docs)
print(f"{len(docs)=}, {len(chunks)=}")

100%|██████████| 2/2 [00:00<00:00, 79.39it/s]


len(docs)=658, len(chunks)=8962


In [ ]:
lens = [len(chunk.page_content.split()) for chunk in chunks]
df = pd.DataFrame(lens)
df

In [ ]:
# dense retrieval
qdrant = QdrantVectorStore.from_documents(
    chunks,
    embeddings,
    url="http://localhost:6333",
    collection_name=collection_name
)

In [ ]:
query = "Comment les responsabilités en matière de contrôle interne sont-elles réparties entre la direction, les unités opérationnelles et les fonctions de supervision (audit, contrôle des risques) au sein de l'EPFL?"
found_docs = qdrant.similarity_search_with_score(query)
found_docs

In [ ]:
# https://medium.com/@animesh.py/harnessing-qdrant-and-langchain-a-step-by-step-integration-guide-0e2c397289b6 -> hybrid retrieval

# TODO : tester d'autres modeles
sparse = FastEmbedSparse(model_name="Qdrant/bm42-all-minilm-l6-v2-attentions")
hybrid_store = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    sparse_embedding=sparse,
    collection_name=collection_name,
    retrieval_mode=RetrievalMode.HYBRID
)

In [ ]:
question = "Pourquoi la commission des prix de la recherche de l'EPFL est composée obligatoirement de 16 membres ?"
dense_q = embeddings.embed_query(question)
sparse_q = sparse.embed_query(question)